<a href="https://colab.research.google.com/github/Vish204/RAG/blob/main/Document_search_github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install PyMuPDF
!pip install sentence-transformers
!pip install faiss-cpu  # or faiss-gpu if you have GPU support
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.4 MB/s eta 0:00:00


In [2]:
import os
import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
from google.colab import files

# Upload multiple files
uploaded_files = files.upload()

# Function to extract text from a single PDF file and split it into paragraphs
def extract_text_from_pdf(pdf_path):
    """Extracts text from a single PDF file and splits it into paragraphs."""
    text = ""
    with fitz.open(pdf_path) as pdf:
        for page in pdf:
            text += page.get_text() + "\n"
    return text.strip().split('\n\n')  # Split into paragraphs


# Extract text from all uploaded PDFs
documents = []
for pdf_file in uploaded_files.keys():
    paragraphs = extract_text_from_pdf(pdf_file)
    documents.append(paragraphs)

print(f" Loaded {len(documents)} PDFs successfully!")

# Load the pre-trained Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model Loaded Successfully!")

# Generate embeddings for all paragraphs
doc_embeddings = []
paragraph_counts = []

for paragraphs in documents:
    embeddings = model.encode(paragraphs)
    doc_embeddings.append(embeddings)
    paragraph_counts.append(len(paragraphs))

doc_embeddings = np.vstack(doc_embeddings)
print("Embedding Shape:", doc_embeddings.shape)

# Build FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

print(f" FAISS Index Built! Total Paragraphs Indexed: {index.ntotal}")


def search_top_k_with_pdf(query, top_k=3):
    """Search relevant paragraphs for a query."""
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, top_k)

    results = []

    for i, idx in enumerate(indices[0]):
        if idx >= 0:
            cumulative_count = 0
            for doc_index, count in enumerate(paragraph_counts):
                if idx < cumulative_count + count:
                    paragraph_index = idx - cumulative_count
                    paragraph = documents[doc_index][paragraph_index]
                    break
                cumulative_count += count

            results.append(
                f"\nDocument {doc_index + 1}, Paragraph {paragraph_index + 1}:\n{paragraph}\n"
            )

    return results


# ----------- SIMPLE QUERY LOOP (replaces widgets) -------------

while True:
    query = input("\nEnter your query (or type 'exit'): ")

    if query.lower() == "exit":
        print("Search ended.")
        break

    results = search_top_k_with_pdf(query, top_k=3)

    print("\n Search Results:")
    print("-" * 60)

    for i, result in enumerate(results, 1):
        print(f"\nTop {i} Match:")
        print(result)
        print("-" * 60)

Saving Dataset1.pdf to Dataset1.pdf
✅ Loaded 1 PDFs successfully!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model Loaded Successfully!
Embedding Shape: (37, 384)
✅ FAISS Index Built! Total Paragraphs Indexed: 37

Enter your query (or type 'exit'): computer applications

🔍 Search Results:
------------------------------------------------------------

Top 1 Match:

Document 1, Paragraph 2:
S. V. Mahadevkar et al.: Review on Machine Learning Styles in Computer Vision—Techniques and Future Directions
FIGURE 1. Segmentation, classification & Object detection.
human intervention to analyze data and spot trends. Machine
learning has a wide range of effects on society, includ-
ing production lines, healthcare, education, transportation,
and food [1]. Machine learning is transforming our lives
and industries in housing and apps, cars, retail, the food
industry, etc.
The goal of machine learning and computer vision is to
impart to computers the ability to gather data, understand it,
and make decisions based on previous and present results.
Computer vision is important for the Internet of Things,
Indust